In [ ]:
# revisemos la data de  merged/causal_model_data.csv
import pandas as pd
df = pd.read_csv("data/clean/merged/causal_model_data.csv")
df

## 1. Filtro de ventana temporal y validación de cobertura

El archivo `causal_model_data.csv` ya viene generado con ventana 2021-2025 desde `00_prep_dataset.ipynb`. Este filtro es una capa de seguridad (defensivo), no una limpieza necesaria hoy — protege contra el día en que `00_prep_dataset` se vuelva a correr con datos más recientes (SIAF y SIEN en crudo ya tienen 2026).

In [ ]:
df = df[df["anio"].between(2021, 2025)].copy()

print(f"Shape tras filtro de años: {df.shape}")
print(f"Años presentes: {sorted(df['anio'].unique())}")
print(f"Distritos únicos: {df['ubigeo'].nunique()}  (debería ser 1,889 — el universo completo)")


## 2. Revisión de nulos

X (contexto territorial) no debería tener ningún nulo — ya se validó en `00_prep_dataset.ipynb`. Los únicos nulos esperados son en las columnas de Y y T, exactamente en las mismas 146 filas (`y_no_disponible` y `t_no_disponible` coinciden siempre, porque `gasto_total_percapita` se calcula dividiendo entre `ninos_evaluados`, no entre población — cuando no hubo niños tamizados ese año, se cae tanto el numerador de Y como el denominador de T).

In [ ]:
nulos = df.isnull().sum()
print("Columnas con nulos:")
print(nulos[nulos > 0])

print()
print(f"y_no_disponible = 1: {df['y_no_disponible'].sum()} filas")
print(f"t_no_disponible = 1: {df['t_no_disponible'].sum()} filas")
print(f"Coinciden exactamente: {((df['y_no_disponible']==1) == (df['t_no_disponible']==1)).all()}")


## 3. Dataset para el modelo Y ~ X ("sin gasto")

Este notebook entrena **solo** el modelo auxiliar de DML que predice `prevalencia_anemia` a partir de X (contexto territorial), sin ver nunca el gasto (T). El residuo de este modelo (Y real - Y predicho) es el insumo que después usará el causal forest.

**Filtrado:** se excluyen las 146 filas con `y_no_disponible = 1` — no se imputa el resultado, simplemente no hay target real contra qué entrenar en esas filas. Esto no borra esas filas del archivo maestro (`causal_model_data.csv` las conserva con su bandera), solo las saca de este entrenamiento puntual.

**`muestra_pequena` (950 filas, Y ruidoso por `ninos_evaluados` < 10):** queda pendiente de decisión — por ahora se conserva en el dataset de entrenamiento sin ponderar ni excluir.

In [ ]:
cols_X = [
    "pct_cultivo", "pct_construido", "pct_desnudo", "pct_agua_visible",
    "n_edificios", "area_construida_m2", "confianza_media",
    "area_distrito_km2", "densidad_edificios_km2",
    "elevacion_media", "pendiente_media",
    "pct_agua_permanente", "pct_agua_estacional",
    "altitude", "superficie", "pob_densidad_2020",
]

anemia_model_df = df[df["y_no_disponible"] == 0].copy()

print(f"Filas excluidas (sin dato de anemia): {(df['y_no_disponible']==1).sum()}")
print(f"Filas para entrenar Y~X: {anemia_model_df.shape[0]}")
print(f"Distritos únicos en el subset de entrenamiento: {anemia_model_df['ubigeo'].nunique()}")
print(f"  -> de los {df['ubigeo'].nunique()} distritos totales, "
      f"{df['ubigeo'].nunique() - anemia_model_df['ubigeo'].nunique()} quedan sin NINGUNA fila con Y disponible")


In [ ]:
X = anemia_model_df[cols_X]
y = anemia_model_df["prevalencia_anemia"]

assert X.isnull().sum().sum() == 0, "X tiene nulos, revisar"
assert y.isnull().sum() == 0, "y tiene nulos, revisar"
assert anemia_model_df["anio"].between(2021, 2025).all(), "Hay años fuera de 2021-2025"

print("X e y sin nulos, ventana 2021-2025 confirmada ✅")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
